# Tutorial 7: Reinforcement Learning (Part 1)
In this tutorial you will apply value iteration and q-learning to train an agent in two problem settings. The first involves an agent navigating a slippery frozen lake environment in order to find treasure and the second is a text-based version of the well-known Flappy Bird game. You will be tasked to complete parts of the code given the algorithms you learnt in the lecture. To gain a deeper understanding of these algorithms you will then play around with different parameters and observe the impact on performance and behaviour of the agent.


## Frozen Lake

![frozen_lake.png](https://www.gymlibrary.dev/_images/frozen_lake.gif)

---

This is a [Toy Text Open AI gym environment](https://www.gymlibrary.dev/environments/toy_text/frozen_lake/) where the agent must traverse the lake and find the treasure whilst avoiding falling into any holes. The state and actions are discrete.

We can instantiate the environment as follows:


```
env = gym.make('FrozenLake-v1', map_name="4x4", is_slippery=True)
```

We can sample actions randomly


### Action Space

*   0: Left
*   1: Down
*   2: Right
*   3: Up

If `is_slippery` is True then the agent will move in intended direction with probability of 1/3 else will move in either perpendicular direction with equal probability of 1/3 in both directions.

For example, if action is left and `is_slippery` is True, then:
- P(move left)=1/3
- P(move up)=1/3
- P(move down)=1/3

### Observation Space
The observation is a value representing the agent’s current position as current_row * nrows + current_col (where both the row and col start at 0). For example, the goal position in the 4x4 map can be calculated as follows: 3 * 4 + 3 = 15. The number of possible observations is dependent on the size of the map. For example, the 4x4 map has 16 possible observations.

###Rewards
*   Reach goal(G): +1
*   Reach hole(H): 0
*   Reach frozen(F): 0

Note: we can query the environment state type via `env.desc`. For example we can check if a particular state is on frozen ice as follows:
```
if env.desc[int(state/env.ncol), int(state % env.ncol)] == b'F':
    print("State ", state, "is on frozen ice!" )
```

### Interacting with the Environment
We can sample actions randomly, get the agent to perform that action and then observe how the environment state changes:
```
state = env.reset()  # this needs to be called once at the start before sending any actions
action = env.action_space.sample()
state, reward, done, info = env.step(action)
```


---



Before we can execute any code we first need to install the following packages:

In [1]:
!pip install gymnasium pyvirtualdisplay pygame > /dev/null 2>&1
!apt-get install -y xvfb python-opengl ffmpeg > /dev/null 2>&1
!apt-get install -y xvfb

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.16).
0 upgraded, 0 newly installed, 0 to remove and 5 not upgraded.


Now import the necessary packages and following helper functions (just execute as is):

In [2]:
import gymnasium as gym
from gymnasium.envs.toy_text.frozen_lake import generate_random_map
import matplotlib.pyplot as plt
from IPython import display as ipythondisplay
from pyvirtualdisplay import Display
from IPython.display import HTML
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
import math
from collections import defaultdict
import pickle
from IPython.display import clear_output

display = Display(visible=0, size=(400, 300))
display.start()

def display_video(frames, framerate=30):
  """Generates video from `frames`.

  Args:
    frames (ndarray): Array of shape (n_frames, height, width, 3).
    framerate (int): Frame rate in units of Hz.

  Returns:
    Display object.
  """
  height, width, _ = frames[0].shape
  dpi = 70
  orig_backend = matplotlib.get_backend()
  matplotlib.use('Agg')  # Switch to headless 'Agg' to inhibit figure rendering.
  fig, ax = plt.subplots(1, 1, figsize=(width / dpi, height / dpi), dpi=dpi)
  matplotlib.use(orig_backend)  # Switch back to the original backend.
  ax.set_axis_off()
  ax.set_aspect('equal')
  ax.set_position([0, 0, 1, 1])
  im = ax.imshow(frames[0])
  def update(frame):
    im.set_data(frame)
    return [im]
  interval = 1000/framerate
  anim = animation.FuncAnimation(fig=fig, func=update, frames=frames,
                                  interval=interval, blit=True, repeat=False)
  return HTML(anim.to_html5_video())

Now let's try instantiate the Frozen Lake environment, execute some random actions and observe the behaviour of the agent (try running a few times to get a different sequence of random actions).

In [3]:
display = Display(visible=0, size=(400, 300))
display.start()

env = gym.make('FrozenLake-v1', map_name="4x4", is_slippery=True, render_mode="rgb_array")
#env = gym.make('FrozenLake-v1', map_name="4x4", is_slippery=True, render_mode="human")  # if you are running gym locally you can remove the "env.render(mode="rgb_array")" lines below and instantiate gym like this and it will render the environment in a pop-up on your screen

state, _ = env.reset()  # this needs to be called once at the start before sending any actions

frames = []
frames.append(env.render())  # so we can visualise things in this notebook we will create a sequence of frames and render them as a video at the end of the episode

for _ in range(50):
    action = env.action_space.sample()
    next_state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated  # Combine both conditions
    frames.append(env.render())
    if done:  # done means agent reached a terminal state - for frozen lake this means agent reached treasure (goal state) or fell in a hole
        break

display_video(frames, 10)



## Markov Decision Process (MDP)

![frozen_lake.png](https://upload.wikimedia.org/wikipedia/commons/thumb/a/ad/Markov_Decision_Process.svg/400px-Markov_Decision_Process.svg.png)

*Example of a simple MDP with three states (green circles) and two actions (orange circles), with two rewards (orange arrows). (source: https://en.wikipedia.org/wiki/Markov_decision_process )*

We can frame the frozen lake problem as an MDP which is a four-tuple ($S$, $A$, $T$, $R$):


*   $S$ is the set of states, called the *state space*.
*   $A$ is the set of actions, called the *action space*.
*   $T(s,a,s') = p(s',r | s, a)$ is the transition probability, the probability that action $a$ in state $s$ will lead to successor state $s'$.
*   $R(s,a,s') = r$ is the immediate reward recieved after transitioning from state $s$ to $s'$ due to action $a$.

We wish to find the **optimal policy ($\pi^*$)** for the agent, in our case the best action given the current state, in order to get the maximimum **discounted expected reward** (in the case of the Frozen Lake - reach the treasure without falling into any holes).

## Value Iteration

The **state-value function ($V(s)$)** is a measure of the expected reward you can receive from any given state  given an MDP and a policy describing which actions an agent takes in each state. Value iteration (VI) computes $V(s)$ and then uses this to derive the optimal policy. The value iteration algorithm works as follows:

<div>
<img src="https://core-robotics.gatech.edu/files/2020/12/Value_Iteration-1-768x421.png" width="700"/>
</div>

*Value iteration algorithm. (source: https://core-robotics.gatech.edu/2021/01/19/bootcamp-summer-2020-week-3-value-iteration-and-q-learning/)*

---

Below we will implement the VI algorithm. Fill in the missing lines of code surrounded by comments.


In [4]:
def get_max_action(env, V, s, gamma):
    """Computes action with maximum discounted expected return.

    Args:
        env: gym object.
        V: state-value function
        s: current state
        gamma: discount rate

    Returns:
        Action with maximum expected return at state s and the expected return value
    """
    expected_vals = np.zeros(env.action_space.n)
    for a in range(env.action_space.n):                 # iterate over every possible action
        #P = np.array(env.env.P[s][a])                   # holds information about states given action a
        P = np.array(env.P[s][a])                   # holds information about states given action a
        for transition in P:                            # iterate over every possible transition from state s given action a (each transition has structure: [T(s,a,s'), s', r, done])
            p = transition[0]                           # Transition Probability p(s',r|s,a)
            s_= int(transition[1])                      # possible successor state s' given action a in state s
            r = transition[2]                           # reward r for landing in s' given action a from state s

            ### caculate discounted expected return for taking action a in s ###
            expected_vals[a] += p*(r+gamma*V[s_])
            ####################################################################

    ################ get action with maximum expected value ####################
    max_action = np.argmax(expected_vals)
    ############################################################################
    return max_action, expected_vals[max_action]

def update_state_value(env, V, s, gamma):
    """Performs an iteration of updating state value V(s)

    Args:
        env: gym object.
        V: state-value function
        s: current state
        gamma: discount rate

    Returns:
        state value V(s)
    """
    max_action, max_val =  get_max_action(env, V, s, gamma)
    V[s]=max_val
    return V[s]



def value_iteration(env, gamma, delta_thresh):
    """Value iteration algorithm for computing the optimal policy

    Args:
        env: gym object.
        gamma: discount rate
        delta_thresh: convergence threshold

    Returns:
        state value function V and optimal policy pi
    """
    V = np.zeros(env.observation_space.n)                   # initialize V to arbitrary value, here "zeros" (line 1)
    delta = np.inf                                          # line 2
    iters = 0
    while delta > delta_thresh:                             # while V hasn't converged to optimal value (line 3)
        delta = 0                                           # line 4
        for s in range(env.observation_space.n):            # iterate over all states (line 5)
            temp = V[s]                                     # store old state value (line 6)
            update_state_value(env, V, s, gamma)            # perform iteration of updating state values (line 7)
            delta = max(delta, abs(temp - V[s]))            # assign the maximum state value change for iteration to delta (line 8)
        if iters % 10 == 0:
            print("iteration: ", iters, ", delta: ", delta)
        iters += 1
    pi = np.zeros((env.observation_space.n))                # initialise arbitrary policy
    for s in range(env.observation_space.n):
        action, max_val = get_max_action(env, V, s, gamma)  # extract optimal policy by assigning action to each state that maximises state value (line 11)
        pi[s] = action                                      # Usually a policy is a distribution over actions. However, since we are interested in the deterministic case we just simply map the best action at a given state.

    return V, pi                                            # return optimal state value funtion and optimal policy

Test your VI implementation on the frozen lake environment. Try the following:
* Different values of `gamma` (discount factor) and observe how the policy behaviour changes.
* Try with `is_slippery` argument set to False. Can you explain why the behaviour difference arises?
* Try with the custom and randomly generated 8x8 map.

In [5]:
gamma = 0.99  # discount factor
delta_thresh = 0.000001  # convergence threshold

############################## 4x4 map #########################################
env = gym.make('FrozenLake-v1', map_name="4x4", is_slippery=True, render_mode="rgb_array")
env = env.unwrapped
################################################################################

################# custom 8x8 map - feel free to create your own! ###############
# custom_map = [
#     "SFFFFFFF",
#     "FFFFFFFF",
#     "FFFFHFFF",
#     "FFFFHFFF",
#     "FFFFFFFF",
#     "FFHFHFFF",
#     "FHFHFFFF",
#     "FFFFFFFG",
# ]
# env = gym.make('FrozenLake-v1', desc=custom_map, is_slippery=True)
################################################################################

############################## random map ######################################
# env = gym.make('FrozenLake-v1', desc=generate_random_map(size=8), is_slippery=True)
################################################################################

env.action_space.seed(0)  # so we get same the random sequence every time
state, _ = env.reset(seed=1)  # this needs to be called once at the start before sending any actions
V, pi = value_iteration(env, gamma, delta_thresh)

# test your policy
state, _ = env.reset()
frames = []
frames.append(env.render())
steps = 0
while True and steps < 100:  # in case policy gets stuck (shouldn't happen if valid path exists and optimal policy learnt)
    action = int(pi[state])  # get action given current state from policy
    state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated  # Combine both conditions
    frames.append(env.render())
    if done:
        break

env.close()
display_video(frames, 10)

iteration:  0 , delta:  0.33333333333333337
iteration:  10 , delta:  0.02425207463874124
iteration:  20 , delta:  0.015539645708294497
iteration:  30 , delta:  0.00912292665761949
iteration:  40 , delta:  0.005198966623460977
iteration:  50 , delta:  0.003925307395415467
iteration:  60 , delta:  0.0026925382714672597
iteration:  70 , delta:  0.0017454279786007776
iteration:  80 , delta:  0.001103909881843279
iteration:  90 , delta:  0.0006902690280711221
iteration:  100 , delta:  0.00042929636911204216
iteration:  110 , delta:  0.00026629911683073715
iteration:  120 , delta:  0.0001649826481383565
iteration:  130 , delta:  0.00010215117106920912
iteration:  140 , delta:  6.322963095062306e-05
iteration:  150 , delta:  3.9132349824333446e-05
iteration:  160 , delta:  2.421704221866605e-05
iteration:  170 , delta:  1.4986204367961609e-05
iteration:  180 , delta:  9.273744004389961e-06
iteration:  190 , delta:  5.7387209614345736e-06
iteration:  200 , delta:  3.551186154748809e-06
iterati

## Q- Learning

Suppose we do not know the underlying transition probabilities - as is usually the case in the real world. Then we cannot use VI and instead we need to actively interact with the environment to try find a good policy. Q-learning is an RL algorithm that learns what is called a **Q-function** which returns a **Q-value**, or a state-action value. In contrast to VI, which simply gives a value for being in particular state, the Q-value describes the value of taking a particular *action* in a particular state. This is how we overcome not knowing the underlying transition function.

<div>
<img src="https://core-robotics.gatech.edu/files/2021/01/Screenshot-from-2021-01-05-10-41-57.png" width="700"/>
</div>
![q_learning_algorithm.png](https://core-robotics.gatech.edu/files/2021/01/Screenshot-from-2021-01-05-10-41-57.png)

*Q-Learning algorithm. (Source: https://core-robotics.gatech.edu/2021/01/19/bootcamp-summer-2020-week-3-value-iteration-and-q-learning/)

---

Below we will implement the Q-Learning algorithm. Fill in the missing lines of code surrounded by comments.

First let's define epsilon-greedy policy which is our behaviour policy (line 4), or in other words our way to balance between exploring new states/ actions and focusing on high value states/ actions. The policy works as follows, if a randomly sampled probability is greater than a predefined value then we greedily take the best action, otherwise we randomly sample an action from the space of all actions.

In [6]:
def epsilon_greedy(env, state, Q, epsilon, episodes, episode):
    """Selects an action to take based on a uniformly random sampled number.
    If this number is greater than epsilon then returns action with the largest
    Q-value at the current state. Otherwise it returns a random action.

    Args:
        env: gym object.
        state: current state
        Q: Q-function. This is a dictionary that is indexed by the state and
           returns an array of Q-values for each action at that state. For example,
           Q[0] will return an array of Q-values for state 0 where the index of
           the array corresponds to the action.
        epsilon: control how often you explore random actions versus focusing on
                 high value state and actions
        episodes: maximum number of episodes (used in other epsilon greedy variant later)
        episode: number of episodes played so far (used in other epsilon greedy variant later)

    Returns:
        Action to be executed for next step.
    """
    if np.random.uniform(0, 1) > epsilon:
        #### return the action with the highest Q value at the given state  ####
        return np.argmax(Q[state])
        ########################################################################
    else:
        return env.action_space.sample()

Now let's define the main part of the algorithm. Fill in the missing lines of code surrounded by comments.

In [7]:
def simulate(env, Q, max_episode_length, epsilon, episodes, episode):
    """Rolls out an episode of actions to be used for learning.

    Args:
        env: gym object.
        Q: state-action value function
        epsilon: control how often you explore random actions versus focusing on
                 high value state and actions
        episodes: maximum number of episodes
        episode: number of episodes played so far

    Returns:
        Dataset of episodes for training the RL agent containing states, actions and rewards.
    """
    D = []
    state, _ = env.reset()                                                     # line 2 - note we don't sample the start state since this is predefined
    done = False
    for step in range(max_episode_length):                                  # line 3
        action = epsilon_greedy(env, state, Q, epsilon, episodes, episode)  # line 4
        next_state, reward, terminated, truncated, info = env.step(action)       # line 5
        done = terminated or truncated  # Combine both conditions
        #### change reward so that a negative reward is given if the agent falls down a hole #######
        if env.desc[int(next_state/env.ncol), int(next_state % env.ncol)] == b'H':  # fell in hole
            reward = -1.0
        ############################################################################################
        D.append([state, action, reward, next_state])                       # line 7
        state = next_state                                                  # line 8
        if done:                                                            # if we fall into a hole or reach treasure then end episode
            break
    return D                                                                # line 10

def q_learning(env, gamma, episodes, max_episode_length, epsilon, step_size):
    """Main loop of Q-learning algorithm.

    Args:
        env: gym object.
        gamma: discount factor - determines how much to value future actions
        episodes: number of episodes to play out
        max_episode_length: maximum number of steps for episode roll out
        epsilon: control how often you explore random actions versus focusing on
                 high value state and actions
        step_size: learning rate - controls how fast or slow to update Q-values
                   for each iteration.

    Returns:
        Q-function which is used to derive policy.
    """
    Q = defaultdict(lambda: np.zeros(env.action_space.n))                       # line 2
    total_reward = 0
    for episode in range(episodes):                                             # slightly different to line 3, we just run until maximum episodes played out
        D = simulate(env, Q, max_episode_length, epsilon, episodes, episode)    # line 4
        for data in D:                                                          # data = [state, action, reward, next_state]  (line 5)
            # print(data)
            ####################### update Q value (line 6) #########################
            state = data[0]
            action = data[1]
            reward = data[2]
            next_state = data[3]
            Q[state][action] = (1 - step_size) * Q[state][action] + step_size * (reward +  gamma * np.max(Q[next_state]))  # line 6
            #########################################################################
            total_reward += reward
            # input()
        if episode % 100 == 0:
            print("average total reward per episode batch since episode ", episode, ": ", total_reward/ float(100))
            total_reward = 0
    return Q  # line 9

Test your Q-learning implementation on the frozen lake environment. Try the following:
* If your agent fails to find a successful policy then try increasing the `episodes` variable.
* By default the frozen lake gym environment assigns a zero reward for falling into a hole. Try assigning a negative reward to see if it speeds up training. If so, why do you think that happens?
* Different values of `gamma` (discount factor), `epsilon`, `step_size` (learning rate), `episodes`, and observe how the policy behaviour changes.
* Try with `is_slippery` argument set to False. Can you explain why the behaviour difference arises?
* Try with the custom and randomly generated 8x8 map.

In [13]:
display = Display(visible=0, size=(400, 300))
display.start()

gamma = 0.95                # discount factor - determines how much to value future actions
episodes = 10000            # number of episodes to play out
max_episode_length = 200    # maximum number of steps for episode roll out
epsilon = 0.2               # control how often you explore random actions versus focusing on high value state and actions
step_size = 0.1             # learning rate - controls how fast or slow to update Q-values for each iteration.

############################## 4x4 map #########################################
env = gym.make('FrozenLake-v1', map_name="4x4", is_slippery=True, render_mode="rgb_array")
env = env.unwrapped
################################################################################

################# custom 8x8 map - feel free to create your own! ###############
# custom_map = [
#     "SFFFFFFF",
#     "FFFFFFFF",
#     "FFFFHFFF",
#     "FFFFHFFF",
#     "FFFFFFFF",
#     "FFHFHFFF",
#     "FHFHFFFF",
#     "FFFFFFFG",
# ]
# env = gym.make('FrozenLake-v1', desc=custom_map, is_slippery=True)
################################################################################

############################## random map ######################################
# env = gym.make('FrozenLake-v1', desc=generate_random_map(size=8), is_slippery=True)
################################################################################
import random
env.action_space.seed(0)  # so we get same the random sequence every time
state, _ = env.reset(seed=0)  # this needs to be called once at the start before sending any actions
Q = q_learning(env, gamma, episodes, max_episode_length, epsilon, step_size)

# test your policy
state, _ = env.reset()
frames = []
frames.append(env.render())
steps = 0
while True and steps < 1000:  # in case policy gets stuck (shouldn't happen if valid path exists and optimal policy learnt)
    ########## policy is simply taking the action with the highest Q-value for given state ##########
    action = np.argmax(Q[state])  # get action given current state from policy
    #################################################################################################
    state, reward, terminated, truncated, info = env.step(action)       # line 5
    done = terminated or truncated  # Combine both conditions
    frames.append(env.render())
    if done:
        break

env.close()
display_video(frames, 10)

average total reward per episode batch since episode  0 :  -0.01
average total reward per episode batch since episode  100 :  -0.98
average total reward per episode batch since episode  200 :  -0.94
average total reward per episode batch since episode  300 :  -0.98
average total reward per episode batch since episode  400 :  -0.96
average total reward per episode batch since episode  500 :  -0.94
average total reward per episode batch since episode  600 :  -1.0
average total reward per episode batch since episode  700 :  -0.94
average total reward per episode batch since episode  800 :  -1.0
average total reward per episode batch since episode  900 :  -0.9
average total reward per episode batch since episode  1000 :  -0.92
average total reward per episode batch since episode  1100 :  -0.96
average total reward per episode batch since episode  1200 :  -0.92


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


average total reward per episode batch since episode  1300 :  -0.96
average total reward per episode batch since episode  1400 :  -0.96
average total reward per episode batch since episode  1500 :  -0.98
average total reward per episode batch since episode  1600 :  -0.96
average total reward per episode batch since episode  1700 :  -1.0
average total reward per episode batch since episode  1800 :  -0.96
average total reward per episode batch since episode  1900 :  -0.96
average total reward per episode batch since episode  2000 :  -0.9
average total reward per episode batch since episode  2100 :  -0.96
average total reward per episode batch since episode  2200 :  -0.98
average total reward per episode batch since episode  2300 :  -0.94
average total reward per episode batch since episode  2400 :  -0.96
average total reward per episode batch since episode  2500 :  -0.94
average total reward per episode batch since episode  2600 :  -0.98
average total reward per episode batch since episo

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Try the following variant of epsilon greedy which starts off almost purely random and then decays to taking more greedy actions over the training period. Note: you just need to execute the following block of code and then you can try running your Q-learning training again.

In [9]:
def epsilon_greedy(env, state, Q, epsilon, episodes, episode):
    EPS_START = 0.9
    EPS_END = 0.05
    EPS_DECAY = episodes
    sample = np.random.uniform(0, 1)
    eps_threshold = EPS_END + (EPS_START - EPS_END) * \
        math.exp(-1. * episode / EPS_DECAY)
    if sample > eps_threshold:
        return np.argmax(Q[state])
    else:
        return env.action_space.sample()

Flappy Bird

In [10]:
!pip install git+https://github.com/fredsukkar/flappy-bird-gym > /dev/null 2>&1

In [12]:
import flappy_bird_gym

env = flappy_bird_gym.make("FlappyBird-v0", audio_on=False)
env = env.unwrapped

frames = []
state = env.reset()

while True:
    action = env.action_space.sample() # for a random action
    next_state, reward, done, info = env.step(action)
    frames.append(np.fliplr(np.rot90(env.render(mode="rgb_array"), 3)))
    plt.imshow(frames[-1])
    if done:
        break

env.close()

display_video(frames)

/usr/local/lib/python3.12/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.12/dist-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.12/dist-packages/gym/core.py:43: DeprecationWarning: WARN: The argument mode in render method is deprecated; use render_mode during environment initialization instead.
See here for more information: https://www.gymlibrary.ml/content/api/
  deprecation(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is d